Proyecto 2: Introducción a la Inteligencia Artificial

Integrantes: Vicente Arechavala, Johan Riveros

Profesor: Gabriel Cabas

### Librerias + base de datos

In [30]:
#Librerias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch

In [31]:
from torchvision.models import resnet50
model = resnet50(weights="IMAGENET1K_V1")

In [32]:
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

from torch.nn import CrossEntropyLoss

In [47]:
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [48]:
train_folder = ImageFolder("./Fruits dataset/train", transform=transform)
val_folder = ImageFolder("./Fruits dataset/val", transform=transform)
test_folder = ImageFolder("./Fruits dataset/test", transform=transform)

train_loader = DataLoader(train_folder, batch_size=32, shuffle=True)
val_loader = DataLoader(val_folder, batch_size=32, shuffle=False)
test_loader= DataLoader(test_folder, batch_size=32, shuffle=False)

print("Clases:", train_folder.class_to_idx)
print(f"Train: {len(train_folder)} | Val: {len(val_folder)} | Test: {len(test_folder)}")

Clases: {'abiu': 0, 'acai': 1, 'acerola': 2, 'ackee': 3, 'ambarella': 4, 'apple': 5, 'apricot': 6, 'avocado': 7, 'banana': 8, 'barbadine': 9, 'barberry': 10, 'betel_nut': 11, 'bitter_gourd': 12, 'black_berry': 13, 'black_mullberry': 14, 'brazil_nut': 15, 'camu_camu': 16, 'cashew': 17, 'cempedak': 18, 'chenet': 19, 'cherimoya': 20, 'chico': 21, 'chokeberry': 22, 'cluster_fig': 23, 'coconut': 24, 'corn_kernel': 25, 'cranberry': 26, 'cupuacu': 27, 'custard_apple': 28, 'damson': 29, 'dewberry': 30, 'dragonfruit': 31, 'durian': 32, 'eggplant': 33, 'elderberry': 34, 'emblic': 35, 'feijoa': 36, 'fig': 37, 'finger_lime': 38, 'gooseberry': 39, 'goumi': 40, 'grape': 41, 'grapefruit': 42, 'greengage': 43, 'grenadilla': 44, 'guava': 45, 'hard_kiwi': 46, 'hawthorn': 47, 'hog_plum': 48, 'horned_melon': 49, 'indian_strawberry': 50, 'jaboticaba': 51, 'jackfruit': 52, 'jalapeno': 53, 'jamaica_cherry': 54, 'jambul': 55, 'jocote': 56, 'jujube': 57, 'kaffir_lime': 58, 'kumquat': 59, 'lablab': 60, 'langsat

In [35]:
for params in model.parameters():
    params.requires_grad_ = False

In [36]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
display(device)
model.to(device)

device(type='cpu')

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Con

In [37]:
model.fc = torch.nn.Sequential(
    torch.nn.Dropout(0.5),
    torch.nn.Linear(model.fc.in_features, 5)
).to(device)
model

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Con

In [38]:
from torch.optim import AdamW
import copy

In [39]:
criterion = CrossEntropyLoss()
optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

EPOCHS = 15
PATIENCE = 4

In [40]:
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val_loss = float("inf")
best_state = None
epochs_no_improve = 0

In [45]:
for epoch in range(EPOCHS):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total

    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)

    val_loss = running_loss / total
    val_acc = correct / total

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    print(f"Epoch {epoch+1}/{EPOCHS} | train_loss={train_loss:.4f} acc={train_acc:.3f} "
          f"| val_loss={val_loss:.4f} acc={val_acc:.3f}")

    # --- early stopping ---
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(model.state_dict())
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"Early stopping en epoch {epoch+1}")
            break

model.load_state_dict(best_state)

IndexError: Target 85 is out of bounds.